# Scrolls (Agent Skills)

**Scrolls** are directory-based skill packages that augment an adventurer's capabilities,
aligned with the [Agent Skills specification](https://agentskills.io/specification).

Each scroll is a folder with a `SKILL.md` file:
```
scrolls/
  code-review/
    SKILL.md          # YAML frontmatter (name, description) + instructions
    scripts/          # optional executable code
    references/       # optional documentation
    assets/           # optional templates, resources
```

Key concepts:
- **Name + Description** — lightweight discovery info (~100 tokens)
- **Instructions** — full prompt text from the SKILL.md body, injected on activation
- **Progressive disclosure** — discovered scrolls show only name + description; activated scrolls inject full instructions

Adventurers pick scrolls by name: `adventurer.pick_scroll("code-review")`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Creating and Loading Scrolls

Create skill directories with `SKILL.md` files, then use `ScrollCatalog` to load them.
Pick scrolls by name — no Python classes needed.

In [ ]:
import tempfile
from pathlib import Path

from guildmaster_ai.adventurers import GeneralHero
from guildmaster_ai.scrolls import ScrollCatalog

# Create a temporary scrolls directory with two skills
scrolls_dir = Path(tempfile.mkdtemp()) / "scrolls"
scrolls_dir.mkdir()

# Skill 1: summarizer
(scrolls_dir / "summarizer").mkdir()
(scrolls_dir / "summarizer" / "SKILL.md").write_text("""\
---
name: summarizer
description: Summarize documents and text. Use when asked to condense or summarize content.
---

When summarizing:
1. Read the full document first.
2. Identify the key points and main arguments.
3. Write a concise summary preserving the original meaning.
4. Keep the summary under 20% of the original length.
""")

# Skill 2: data-cleaner (with a script)
(scrolls_dir / "data-cleaner").mkdir()
(scrolls_dir / "data-cleaner" / "SKILL.md").write_text("""\
---
name: data-cleaner
description: Clean and normalize datasets. Use for CSV/JSON data preprocessing.
---

To clean a dataset:
1. Identify column types and missing values.
2. Run `scripts/clean.py <input> <output>` to apply standard transformations.
3. Report what was changed.
""")
scripts_dir = scrolls_dir / "data-cleaner" / "scripts"
scripts_dir.mkdir()
(scripts_dir / "clean.py").write_text("# placeholder cleaning script\n")

# Load the catalog and pick scrolls by name
catalog = ScrollCatalog(scrolls_dir)
print(f"Available scrolls: {catalog.list_names()}")

# Heroes (BaseHero / GeneralHero) support scrolls — regular adventurers do not
hero = GeneralHero(name="Analyst", scroll_catalog=catalog)
hero.pick_scroll("summarizer")
hero.pick_scroll("data-cleaner")

print(f"\nHero scrolls: {hero.scroll_names}")

## Scroll Catalog

The `ScrollCatalog` scans a directory for skill subdirectories and provides name-based lookup.
Heroes pick scrolls by name — scroll directories are then passed as native skills to the
deep agent at execution time.

In [ ]:
scout = GeneralHero(name="Scout", scroll_catalog=catalog)

# Pick specific scrolls — their directories are passed as skills to the deep agent
scout.pick_scroll("summarizer")
print(f"Scout scrolls: {scout.scroll_names}")

# Scrolls can be listed from the catalog without picking
print(f"\nAll available scrolls in catalog: {catalog.list_names()}")

## Using with GuildBuilder

The `GuildBuilder` can configure a scrolls directory so all registered adventurers
get access to the catalog automatically.

```python
guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .with_scrolls_dir("./scrolls")       # all adventurers can pick_scroll("name")
    .register_adventurer(my_adventurer)
    .build()
)
```

## Scroll with Script Execution (End-to-End Quest)

This example shows a scroll that bundles a Python script (`scripts/profile_csv.py`).
The SKILL.md instructions tell the agent to run the script via the `run_script` weapon,
then summarize the results.

The built-in `data-profiler` skill ships with guildmaster-ai at
`guildmaster_ai/scrolls/skills/data-profiler/`.

In [ ]:
import csv
import tempfile
from pathlib import Path

from guildmaster_ai import GeneralHero, GuildBuilder
from guildmaster_ai.scrolls import ScrollCatalog

# ── 1. Locate the built-in skills directory ──────────────────────────
skills_dir = Path("../guildmaster_ai/scrolls/skills")
catalog = ScrollCatalog(skills_dir)
print(f"Built-in scrolls: {catalog.list_names()}")

# Inspect the data-profiler skill
profiler = catalog.get("data-profiler")
print(f"\n--- SKILL.md for '{profiler.name}' ---")
print(profiler.instructions)

In [ ]:
# ── 2. Create a sample CSV to profile ────────────────────────────────
csv_path = Path(tempfile.mktemp(suffix=".csv"))
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "salary", "department"])
    writer.writerows([
        ["Alice", "32", "85000", "Engineering"],
        ["Bob", "28", "", "Marketing"],
        ["Charlie", "45", "120000", "Engineering"],
        ["Diana", "", "95000", "Sales"],
        ["Eve", "35", "78000", ""],
    ])
print(f"Sample CSV: {csv_path}")
print(csv_path.read_text())

In [ ]:
# ── 3. Set up hero with scroll, run the quest ────────────────────────

# Deep agents (BaseHero/GeneralHero) handle script execution natively
# via their FilesystemBackend — no need to equip a ScriptRunWeapon.
analyst = GeneralHero(name="Analyst", scroll_catalog=catalog)
analyst.pick_scroll("data-profiler")

print(f"Weapons: {analyst.weapon_names}")
print(f"Scrolls: {analyst.scroll_names}")

# Build guild and run
guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .with_scrolls_dir(skills_dir)
    .register_adventurer(analyst)
    .build()
)

result = await guild.post_quest(
    f"Profile the CSV file at {csv_path} and tell me about any data quality issues."
)
print("\n=== Quest Result ===")
print(result.summary)